In [11]:
import pandas as pd
from typing import Dict, List, Tuple
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain, SequentialChain
from OprFuncs import *
#from langchain.schema.runnable import RunnableSequence
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
#from langchain.agents import AgentExecutor, Tool, create_react_agent
#from langchain import hub
import re
#from modelEXT.PygalCodeComponents import PygalCodeComponents
#from langchain.output_parsers import PydanticOutputParser
from DatabaseManager import DatabaseManager
from langchain_experimental.agents import create_pandas_dataframe_agent

class DataAnalyzer:
    def __init__(self,dataframe,llm,user_id=None):
        self.dataframe = dataframe
        self.llm = llm
        self.data_info = data_infer(dataframe)
        self.data_description = data_describer(dataframe)
        self.data_sample = dataframe.head().to_string()
        self.data_cols = ", ".join(dataframe.columns)
        self.db = DatabaseManager()
        self.report_id = None
        self.memory = []
        
        if user_id:
            self.user_id = user_id
            self.user_context = self.db.get_user_context(user_id)
            if self.user_context:
                self.memory.append(HumanMessage(content=f"User Context: {self.user_context}"))
        else:
            self.user_context = None

    def analysis_data(self):
        data_info = self.data_info
        data_sample = self.data_sample
        data_description = self.data_description

        analysis_template = '''
        You are a highly skilled professional data analyst specialized in business data analysis.

        You are provided with:
        1. Dataset metadata: {data_info}
        2. Dataset sample: {data_sample}
        3. Dataset summary: {data_description}
        4. User context: {user_context}

        Your task is to provide a structured and insightful business data analysis report.

        Your analysis must:
        - Uncover patterns, trends, and key insights.
        - Highlight any surprising, concerning, or high-impact findings.
        - Identify strengths, weaknesses, opportunities, and risks.
        - Connect findings to business strategy and decision-making.
        - Use professional, executive-level language.
        - Be clear, structured, and insightful.
        - Reference actual metrics and statistics where possible.

        ---

        ### 📊 Executive Summary

        - Provide a concise overview of the most important findings and what they mean for the business.

        ---

        ### 📈 Key Patterns & Insights

        1. **🏠 Home Team Goals: A Trend to Build Upon**  
        The average number of home team goals per match is increasing over time (**mean: 1.811**). This suggests improved performance and presents opportunities for increased revenue via ticket sales, sponsorships, and merchandise.

        2. **🎟️ Attendance: A Correlation Worth Exploring**  
        A strong positive correlation exists between attendance and home team goals (**R² = 0.75**). More engaged crowds appear to boost home team performance — highlighting the need to enhance the match-day experience.

        3. **⏱️ Half-time Home Goals: A Key Indicator**  
        The average number of half-time home goals is rising (**mean: 0.708**), signaling strong starts and potentially higher match finishes. This has implications for fantasy sports and sports betting engagement.

        ---

        ### ⚠️ Risks, Challenges & Weaknesses

        1. **📉 Attendance Fluctuations**  
        A moderate negative correlation between attendance and away team goals (**R² = -0.45**) implies weaker away team performance may deter fans. This could negatively affect revenue, especially in less competitive matchups.

        2. **⚖️ Data Imbalance**  
        75% of matches are skewed toward teams with higher average goals. This imbalance could bias predictive models. Consider resampling strategies like oversampling or undersampling.

        ---

        ### 🌱 Opportunities for Growth

        1. **🎊 Optimize Match-Day Experience**  
        Leverage the strong attendance-home goal correlation to invest in fan zones, entertainment, and stadium experiences that drive engagement and revenue.

        2. **🧠 Enhance Fantasy Sports Offerings**  
        Rising trends in home and half-time goals indicate an opportunity to build more engaging and dynamic fantasy game formats.

        ---

        ### 🔍 Additional Insights

        1. **🔁 Unexpected Correlation**  
        A **moderate positive correlation** (**R² = 0.25**) exists between attendance and **away team goals** — suggesting fans are also drawn to exciting high-scoring games, regardless of team allegiance. This opens up alternative marketing narratives.

        2. **🔬 Deeper Analytical Paths**  
        Further investigation into possession, shot accuracy, and team structure can reveal micro-level levers to inform business and coaching strategies.

        ---

        ### 🧠 Strategic Reflection

        - Based on these insights, the organization should prioritize enhancing the fan experience both in-stadium and digitally, while also preparing predictive models with proper data balancing techniques.
        - Ensure data-driven decision-making incorporates both trends and outliers.
        - Beware of biases caused by data imbalance and continuously validate models against new seasons or competitions.

        ---

        *Prepared by your Data Analyst Agent*
        '''

        analysis_prompt = PromptTemplate(
            input_variables=["data_info", "data_sample", "data_description", "user_context"],
            template=analysis_template
        )
        
        analysis_chain = analysis_prompt | self.llm

        self.analysis = analysis_chain.invoke({
            "data_info": data_info,
            "data_sample": data_sample,
            "data_description": data_description,
            "user_context":self.user_context or "No prior context available"
        })

        formatted_analysis_prompt = analysis_template.format(data_info=data_info,data_sample=data_sample,
                                                            data_description=data_description,
                                                            user_context=self.user_context)
        self.memory.append(HumanMessage(content=formatted_analysis_prompt))
        self.memory.append(AIMessage(content=self.analysis))
        self.db.saveMemory(reportID=self.report_id,
                        llm=self.db.llm_id_by_name(self.llm.model),
                        prompet=formatted_analysis_prompt,
                        response=self.analysis,
                        chat=False)
        self.generate_user_context()
        return self.analysis
    
    def questions_gen(self, num):
        data_info = self.data_info
        data_sample = self.data_sample
        data_description = self.data_description

        question_prompt = f"""
        You are a senior data analyst hired by a company to extract meaningful, high-level, and actionable business insights from the following dataset.

        Your job is to generate advanced **strategic questions** that:
        - Are deeply rooted in the data structure and semantics.
        - Reflect important **business objectives**, patterns, risks, or growth opportunities.
        - Are **strong, insightful, and relevant** to decision-makers like company owners or managers.
        - Can be **easily visualized** using bar charts, line plots, histograms, scatter plots, or pie charts.

        **DO NOT generate general or surface-level questions. Instead, focus on questions that:**
        - Quantify change over time or between groups.
        - Explore distribution, frequency, or correlation.
        - Investigate trends, seasonality, or anomalies.
        - Provide guidance for optimizing business performance or identifying risks.

        You MUST generate exactly {num} chartable, insightful questions.

        ### INPUTS:
        1. Dataset Overview: {data_info}
        2. Dataset Sample: {data_sample}
        3. Data Summary: {data_description}
        4. Business Context: {self.user_context}

        ### OUTPUT FORMAT:
        Write {num} powerful analytical questions that:
        - Could be visualized with a chart.
        - Have clear business relevance.
        - Reflect advanced reasoning.

        Each question should be written on a separate line.

        Example Questions:
        - How has the conversion rate changed over time across different marketing channels?
        - Which regions have shown the fastest growth in revenue over the past year?
        - What is the correlation between customer satisfaction scores and return frequency?
        - How does the average transaction value vary by customer segment?
        """

        question_template = PromptTemplate(
            input_variables=["num", "data_info", "data_sample", "data_description"],
            template=question_prompt
        )

        question_chain = question_template | self.llm

        try:
            generated_questions = question_chain.invoke({
                "num": num,
                "data_info": data_info,
                "data_sample": data_sample,
                "data_description": data_description,
                "user_context": self.user_context
            })

            # Ensure the response is properly encoded
            if isinstance(generated_questions, str):
                generated_questions = generated_questions.encode('utf-8', 'replace').decode('utf-8')

            print("Raw LLM Output:", repr(generated_questions))

            if not generated_questions.strip():
                print("Warning: LLM did not generate any questions.")
                return []

            # Use the improved extraction function
            questions_list = extract_questions(generated_questions)

            print("Extracted Questions List:", questions_list)

            if len(questions_list) > num:
                questions_list = questions_list[:num]
            elif len(questions_list) < num:
                print(f"Warning: Expected {num} questions, but got {len(questions_list)}")

            # Store in memory
            formatted_question_prompt = question_template.format(
                num=num,
                data_info=data_info,
                data_sample=data_sample,
                data_description=data_description
            )
            self.memory.append(HumanMessage(content=formatted_question_prompt))
            self.memory.append(AIMessage(content="\n".join(questions_list)))
            self.db.saveMemory(
                reportID=self.report_id,
                llm=self.db.llm_id_by_name(self.llm.model),
                prompet=formatted_question_prompt,
                response="\n".join(questions_list),
                chat=False
            )

            return questions_list

        except Exception as e:
            print(f"Error generating questions: {str(e)}")
            return []

        
    def generate_recommendations(self, num_recommendations: int = 5):
        data_info = self.data_info
        data_sample = self.data_sample
        data_description = self.data_description
        analysis = self.analysis  # التحليل الذي تم عمله سابقاً

        recommendation_prompt = '''
        You are a world-class business consultant and data analyst.

        You have analyzed the following:
        - Dataset metadata: {data_info}
        - Dataset sample: {data_sample}
        - Dataset summary: {data_description}
        - Detailed business analysis: {analysis}
        - User context: {user_context}

        Based on your deep understanding of the data and analysis:
        Your task is to generate {num_recommendations} highly actionable, strategic recommendations for the business.

        Your recommendations must:
        - Be directly based on the analysis and insights.
        - Address clear business actions (e.g., optimize processes, launch new products, reduce risks, target specific segments, etc.)
        - Be specific, impactful, and feasible.
        - Cover both short-term quick wins and long-term strategic moves.
        - Include estimated expected outcome in percentage (%) where appropriate.
        - Include any potential risks or challenges for each recommendation.
        - Reference relevant metrics or insights from the analysis if possible.
        - Use professional, executive-level language.
        - Add an appropriate emoji based on risk level:
            - ✅ for Low risk
            - ⚠️ for Medium risk
            - ❗for High risk

        Output Format:

        ### 📋 Recommendations Table

        | # | Recommendation Title | Expected Impact (%) | Potential Risk (with Emoji) |
        |---|-----------------------|---------------------|-----------------------------|
        | 1 | [Title] | [Estimated Impact %] | [Emoji] [Main risk] |
        | 2 | [Title] | [Estimated Impact %] | [Emoji] [Main risk] |
        | ... | ... | ... | ... |

        ---

        ### 📋 Full Recommendation Details

        1. **[Recommendation Title]** [Emoji]
        - **Details:** Explain clearly what should be done and why.
        - **Expected Impact:** [e.g., Increase attendance by 10%]
        - **Metrics Reference:** [Reference specific metric if available, e.g., matches with <50% attendance]
        - **Potential Risks:** [Possible challenges or risks involved]
        - **Timeline:** [Short-term or Long-term]

        Repeat similarly for each recommendation.
        '''

        
        rec_template = PromptTemplate(
            input_variables=["data_info", "data_sample", "data_description", "analysis", "user_context", "num_recommendations"],
            template=recommendation_prompt
        )

        rec_chain = LLMChain(llm=self.llm, prompt=rec_template)

        rec_response = rec_chain.run(
            data_info=data_info,
            data_sample=data_sample,
            data_description=data_description,
            analysis=analysis,
            user_context=self.user_context or "No prior context available",
            num_recommendations=num_recommendations
        )

        # تسجيل في الذاكرة
        formatted_rec_prompt = recommendation_prompt.format(
            data_info=data_info,
            data_sample=data_sample,
            data_description=data_description,
            analysis=analysis,
            user_context=self.user_context or "No prior context available",
            num_recommendations=num_recommendations
        )
        self.memory.append(HumanMessage(content=formatted_rec_prompt))
        self.memory.append(AIMessage(content=rec_response))
        self.db.saveMemory(reportID=self.report_id,
                        llm=self.db.llm_id_by_name(self.llm.model),
                        prompet=formatted_rec_prompt,
                        response=rec_response,
                        chat=False)

        return rec_response


In [10]:
import pandas as pd
from DataAnalyzer import DataAnalyzer 
from langchain_ollama import OllamaLLM

# Load the data
data = pd.read_csv("WorldCupMatches/WorldCupMatches.csv")
df = pd.DataFrame(data)

# Load the model
llm = OllamaLLM(model='llama3')

# Create analyzer object
analyzer = DataAnalyzer(dataframe=df, llm=llm, user_id='huss')

# Run analysis
analysis_result = analyzer.analysis_data()

# Print the analysis
print(analysis_result)

### 📊 Executive Data Analysis Report

#### 🔍 Key Findings

1. **Most Home Teams Score 2-3 Goals Per Match**: With an average of 1.81 goals scored by home teams, it's clear that many are able to capitalize on their home advantage. This finding has significant implications for sports betting and team strategy, as it highlights the importance of defending at home.

Why: Understanding this trend can inform decisions around fixture scheduling, player selection, and defensive tactics.

2. **Attendance is Highly Varied**: The mean attendance is 45,164, but with a standard deviation of 23,485, there's significant variation in crowd sizes across matches. This could be attributed to factors like team popularity, stadium capacity, or external factors like weather or competition.

Why: Recognizing this trend can help event organizers and teams better understand their audience and optimize ticket sales and marketing efforts.

3. **Half-time Away Goals are Lower Than Expected**: With an average of 0

In [12]:
import pandas as pd
from DataAnalyzer import DataAnalyzer 
from langchain_ollama import OllamaLLM

# Load the data
data = pd.read_csv("WorldCupMatches/WorldCupMatches.csv")
df = pd.DataFrame(data)

# Load the model
llm = OllamaLLM(model='llama3')

# Create analyzer object
analyzer = DataAnalyzer(dataframe=df, llm=llm, user_id='huss')

# Run analysis
analysis_result = analyzer.analysis_data()

# Print the analysis
print(analysis_result)

**Executive Data Analysis Report**

### 🔍 Key Findings
1. **Home Team Goals vs. Attendance**: A strong correlation exists between home team goals scored and attendance numbers (R-squared = 0.74). This suggests that when the home team scores more goals, they tend to attract larger crowds. For businesses, this means that a winning strategy can lead to increased customer engagement and revenue.
2. **Half-time Score Impact**: The half-time score has a significant impact on the overall match outcome (p-value < 0.05). Teams that are ahead at halftime have a higher chance of winning (70%), while teams trailing at halftime tend to lose (65%). This finding highlights the importance of strong team performance during critical periods in a game, mirroring real-world business strategies where early gains can set the stage for long-term success.
3. **Referee Impact**: An analysis of referee assignments reveals that certain referees have distinct tendencies towards home teams or away teams (p-value <

In [ ]:
import pandas as pd
from DataAnalyzer import DataAnalyzer 
from langchain_ollama import OllamaLLM

# Load the data
data = pd.read_csv("WorldCupMatches/WorldCupMatches.csv")
df = pd.DataFrame(data)

# Load the model
llm = OllamaLLM(model='llama3')

# Create analyzer object
analyzer = DataAnalyzer(dataframe=df, llm=llm, user_id='huss')

# Run analysis
analysis_result = analyzer.questions_gen(num=5)

# Print the analysis
print(analysis_result)


Raw LLM Output: 'Here are five analysis questions about the dataset:\n\n1. What is the average attendance by stage in the tournament?\n2. Which team scored the most goals during half-time, and what was their overall record?\n3. Is there a correlation between home team goals and win conditions?\n4. How does the distribution of away team goals vary across different cities (Montevideo)?\n5. What is the relationship between RoundID and MatchID?'
Extracted Questions List: ['What is the average attendance by stage in the tournament?', 'Which team scored the most goals during half-time, and what was their overall record?', 'Is there a correlation between home team goals and win conditions?', 'How does the distribution of away team goals vary across different cities (Montevideo)?', 'What is the relationship between RoundID and MatchID?']
['What is the average attendance by stage in the tournament?', 'Which team scored the most goals during half-time, and what was their overall record?', 'Is th

In [17]:
import pandas as pd
from DataAnalyzer import DataAnalyzer 
from langchain_ollama import OllamaLLM

# Load the data
data = pd.read_csv("WorldCupMatches/WorldCupMatches.csv")
df = pd.DataFrame(data)

# Load the model
llm = OllamaLLM(model='llama3')

# Create analyzer object
question = DataAnalyzer(dataframe=df, llm=llm, user_id='huss')

# Run analysis
question_result = question.questions_gen(num=5)

# Print the analysis
print(question_result)

Raw LLM Output: 'Here are five analysis questions about the dataset:\n\n1. What is the average attendance by year?\n2. Are there any significant differences in home team goals scored between different stages (Group 1, Group 2, etc.)?\n3. How does the distribution of half-time home goals compare to that of away goals?\n4. Is there a correlation between win conditions and match outcome (win/loss)?\n5. What is the average attendance for matches with high-scoring games (i.e., games with more than 3 total goals)?'
Extracted Questions List: ['What is the average attendance by year?', 'Are there any significant differences in home team goals scored between different stages (Group 1, Group 2, etc.)?', 'How does the distribution of half-time home goals compare to that of away goals?', 'Is there a correlation between win conditions and match outcome (win/loss)?', 'What is the average attendance for matches with high-scoring games (i.e., games with more than 3 total goals)?']
['What is the averag

In [ ]:
import pandas as pd
from DataAnalyzer import DataAnalyzer 
from langchain_ollama import OllamaLLM

# Load the data
data = pd.read_csv("WorldCupMatches/WorldCupMatches.csv")
df = pd.DataFrame(data)

# Load the model
llm = OllamaLLM(model='llama3')

# Create analyzer object
recommandation = DataAnalyzer(dataframe=df, llm=llm, user_id='huss')

# Run analysis first
recommandation.analysis_data()

# Then generate recommendations
recommandation_result = recommandation.generate_recommendations(5)

# Print the recommendations
print(recommandation_result)


d:\My-Githup\Axiora\DataAnalyzer.py:443: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  rec_chain = LLMChain(llm=self.llm, prompt=rec_template)
d:\My-Githup\Axiora\DataAnalyzer.py:445: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  rec_response = rec_chain.run(


### 📋 Recommendations Table

| # | Recommendation Title | Expected Impact (%) | Potential Risk (with Emoji) |
|---|-----------------------|---------------------|-----------------------------|
| 1 | Optimize Home Team Performance | 15% increase in home team wins | ✅ Low risk of overestimating opponents |
| 2 | Enhance Away Team Strategy | 8% increase in away team points scored | ⚠️ Medium risk of losing key players |
| 3 | Launch Fan Engagement Campaign | 12% increase in attendance for high-profile matches | ❗ High risk of alienating existing fans |
| 4 | Implement Data-Driven Coaching | 10% increase in win rate for top-performing teams | ✅ Low risk of misinterpreting data insights |
| 5 | Invest in Player Development Program | 11% increase in player value over the next two seasons | ⚠️ Medium risk of not attracting top talent |

### 📋 Full Recommendation Details

1. **Optimize Home Team Performance** ✅
- **Details:** Develop a customized coaching strategy for each home team, focusing o

In [2]:
import pandas as pd
from DataAnalyzer import DataAnalyzer 
from langchain_ollama import OllamaLLM

# Load the data
data = pd.read_csv("WorldCupMatches/WorldCupMatches.csv")
df = pd.DataFrame(data)

# Load the model
llm = OllamaLLM(model='llama3') 

# Create analyzer object
question = DataAnalyzer(dataframe=df, llm=llm, user_id='huss')

# Run analysis to generate strong business questions
question_result = question.questions_gen(num=5)

# Print the analysis results
print("Generated Analytical Questions:")
for i, q in enumerate(question_result, 1):
    print(f"{i}. {q}")

Raw LLM Output: 'Here are 5 chartable, insightful questions that reflect advanced reasoning:\n\n1. **How has the distribution of home team goals changed over time?** \nVisualize: Bar chart or histogram\nBusiness Relevance: Analyzing goal distribution can help identify trends in team performance and inform strategies for improving scoring.\n\n2. **What is the relationship between attendance and half-time scores?**\nVisualize: Scatter plot or regression line\nBusiness Relevance: Understanding how crowd size affects team momentum can inform decisions on ticket pricing, promotions, and fan engagement initiatives.\n\n3. **Are there any significant differences in home team goals scored during day versus night matches?**\nVisualize: Bar chart or box plot\nBusiness Relevance: Identifying potential biases in schedule timing (day vs. night) can help teams adjust their strategies for optimal performance.\n\n4. **How do home and away team performance metrics vary by stage of the competition?** \nV